# Student Dropout Prediction Notebook

This notebook reproduces your script as a step-by-step workflow:
1. Load libraries and dataset
2. Clean and preprocess target labels
3. Visualize correlations and class-wise distributions
4. Train and evaluate a Random Forest classifier
5. Compare split seeds and cross-validation results
6. Plot normalized confusion matrix
7. Retrain using only 1% of training data

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import plot_tree


## Question 1 
Load the dataset and display basic information

In [ ]:
# Question 1
# Load the dataset and display basic information about it
df = pd.read_csv('data.csv', sep=';')
df.info()

## Question 2
1. Visualise the data to see if any features are redundant and the overall distribution of the features within the two classes
2. Interpret the results

In [ ]:
# Clean column names in case they contain extra whitespace or quotes
df.columns = df.columns.str.strip().str.replace('"', '')

# Map Target to binary: 0=Dropout, 1=Enrolled/Graduate
target_mapping = {'Dropout': 0, 'Enrolled': 1, 'Graduate': 1}
df['Target'] = df['Target'].map(target_mapping)

# Drop potential NaNs after mapping
df = df.dropna(subset=['Target'])

# Divide the dataset into features and target variable
features = df.drop('Target', axis=1) 
target = df['Target']

# Display class distribution
print(target.value_counts().rename(index={0: 'Dropout (0)', 1: 'Success (1)'}))

In [ ]:
# Correlation matrix to detect redundant features
plt.figure(figsize=(12, 10))
corr_matrix = features.corr()
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Matrix (Check for Redundancy)')
plt.show()

In [ ]:
# Normalize features for visualization
scaler = StandardScaler()
features_scaled = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)
features_scaled['Target'] = target.reset_index(drop=True).map({0: 'Dropout', 1: 'Success'})

# Show boxplots in groups of 7 features to see data distribution by class
features_per_plot = 6

for i, start in enumerate(range(0, len(features.columns), features_per_plot), start=1):
    chunk = features.columns[start:start + features_per_plot]
    # melt convertit le tableau en format long (Feature, Scaled Value, Target), qui est le format attendu par boxplot.
    data = features_scaled.melt(
        id_vars='Target',
        value_vars=chunk,
        var_name='Feature',
        value_name='Scaled Value'
    )

    plt.figure(figsize=(15, 5))
    sns.boxplot(data=data, x='Feature', y='Scaled Value', hue='Target', palette='Set2')
    plt.title(f'Normalized Feature Distributions - Group {i}')
    plt.xticks(rotation=35, ha='right')
    plt.show()

features_scaled = features_scaled.drop(columns='Target')

## Question 3
1. Split the dataset into a training set (90%) et a test set (10%)
2. Train a classifier using the algorithm of your choice
3. Calculate the classifier accuracy
4. Explain the algorithm
5. Where does the difference between the training and test accuracy come from?

In [ ]:
# Split dataset 90/10
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.10, random_state=42)

# Train Random Forest
clf = RandomForestClassifier(random_state=42)
clf.fit(features_train, target_train)
one_tree = clf.estimators_[0]
plt.figure(figsize=(12, 8))
plot_tree(one_tree, filled=True, feature_names=features.columns, class_names=['Dropout (0)', 'Success (1)'])
plt.show()

train_acc = accuracy_score(target_train, clf.predict(features_train))
test_acc = accuracy_score(target_test, clf.predict(features_test))

print(f'Training Accuracy: {train_acc:.4f}')
print(f'Testing Accuracy: {test_acc:.4f}')

# Optional full report
print('\nClassification Report (test split):')
print(classification_report(target_test, clf.predict(features_test), target_names=['Dropout (0)', 'Success (1)']))

#### Explain the algorithm
The Random Forest algorithm builds a lot of decision trees (100 in this case), each trained differently, and makes them vote together. The final prediction is the majority answer across all trees.

Two mechanisms make each tree different from the others :
- Bagging : each tree is trained on a random subset of the data (with replacement)
- Feature randomness : at each split, each tree only considers a random subset of variables — not all of them

#### Where does the difference between the training and test accuracy come from?
Because the classifier is trained with the values of the training set, it is optimized to classify the data from that set correctly. The test set has data that the model has never seen.

The main cause is overfitting. Overfitting is when the model learns the training data too well, instead of learning general patterns.

## Question 4
1. Change the random seed of the train/test split
2. Why is the result different
3. Do a more through evaluation aggregating several different splits

In [ ]:
# Change random seed and evaluate
X_train_seed, X_test_seed, y_train_seed, y_test_seed = train_test_split(
    X, y, test_size=0.10, random_state=99
)

clf_seed = RandomForestClassifier(random_state=42)
clf_seed.fit(X_train_seed, y_train_seed)

print(f'Testing Accuracy with seed 99: {accuracy_score(y_test_seed, clf_seed.predict(X_test_seed)):.4f}')

# 10-fold cross-validation
cv_scores = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=10)
print(f'10-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})')

In [ ]:
# Normalized confusion matrix
y_pred = clf.predict(X_test)
cm_normalized = confusion_matrix(y_test, y_pred, normalize='true')

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_normalized,
    annot=True,
    cmap='Blues',
    xticklabels=['Dropout (0)', 'Success (1)'],
    yticklabels=['Dropout (0)', 'Success (1)']
)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Normalized Confusion Matrix')
plt.show()

In [ ]:
# Retrain using only 1% of the training set
# 1% of the 90% train split = 0.9% of total dataset
X_train_1pct, _, y_train_1pct, _ = train_test_split(
    X_train, y_train, train_size=0.01, random_state=42, stratify=y_train
)

clf_1pct = RandomForestClassifier(random_state=42)
clf_1pct.fit(X_train_1pct, y_train_1pct)

test_acc_1pct = accuracy_score(y_test, clf_1pct.predict(X_test))
print(f'Testing Accuracy (trained on 1% data): {test_acc_1pct:.4f}')